# Task 1 — Standalone Universal Multi-Corruption Denoising Autoencoder

A self-contained notebook for the assignment's Task 1. It trains one blind restoration model on clean inputs, salt-and-pepper noise, Gaussian blur, and rectangular occlusion. Corruptions are created at runtime; validation and test settings are deterministic and saved as manifests. The notebook includes the required Optuna search, final training, per-corruption/per-severity metrics, visual error maps, and failure-case inspection.

The notebook reads the repository's official Oxford-IIIT Pet split and local image files, but imports no project-specific Python modules.

## 1. Configuration and imports

In [ ]:
from pathlib import Path
import json, math, random, time, copy
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torchmetrics.functional import structural_similarity_index_measure
import optuna

# Locate this repository whether Jupyter starts at the repo root or in research/notebooks.
_here = Path.cwd().resolve()
_candidates = [_here, *_here.parents]
REPO_ROOT = next((p for p in _candidates if (p / "research/datasets/oxford-iiit-pet/images").is_dir()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not locate research/datasets/oxford-iiit-pet/images")
RESEARCH_ROOT = REPO_ROOT / "research"
IMAGE_DIR = RESEARCH_ROOT / "datasets/oxford-iiit-pet/images"
SPLIT_PATH = RESEARCH_ROOT / "split_manifest_official.json"
if not SPLIT_PATH.is_file():
    raise FileNotFoundError(f"Official split manifest missing: {SPLIT_PATH}")
with SPLIT_PATH.open() as f:
    SPLITS = json.load(f)

IMAGE_SIZE = 128
SEED = 42
BATCH_SIZE = 32
BASE_CHANNELS = 32
LATENT_DIM = 256
DROPOUT = 0.1
ALPHA = 0.8
LR = 3e-4
N_TRIALS = 25
SEARCH_EPOCHS = 5
FINAL_EPOCHS = 60
NUM_WORKERS = 2
TINY_RUN = False  # Set True for a quick pipeline smoke run; not for reported results.
TINY_SIZE = 64
if TINY_RUN:
    N_TRIALS, SEARCH_EPOCHS, FINAL_EPOCHS = 2, 1, 2

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("Device:", device, "| Dataset:", IMAGE_DIR)
print({k: len(v) for k, v in SPLITS.items()})

## 2. Runtime corruption pipeline and datasets

Training chooses each of the four input conditions with equal probability and samples severity at image load time. Validation is fixed by a saved manifest. The test manifest contains clean plus low, medium, and high severities for each corruption.

In [ ]:
CLASSES = ("clean", "salt", "blur", "occlusion")
BENCHMARKS = {
    "clean": [{"severity": "none"}],
    "salt": [
        {"severity": "low", "prob": 0.03}, {"severity": "medium", "prob": 0.08}, {"severity": "high", "prob": 0.15}],
    "blur": [
        {"severity": "low", "kernel_size": 3, "sigma": 0.7},
        {"severity": "medium", "kernel_size": 5, "sigma": 1.5},
        {"severity": "high", "kernel_size": 7, "sigma": 2.5}],
    "occlusion": [
        {"severity": "low", "num_rects": 1, "coverage": 0.10},
        {"severity": "medium", "num_rects": 2, "coverage": 0.20},
        {"severity": "high", "num_rects": 3, "coverage": 0.35}],
}

def image_tensor(filename):
    with Image.open(IMAGE_DIR / filename) as im:
        arr = np.asarray(im.convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE)), dtype=np.float32) / 255.0
    return torch.from_numpy(arr).permute(2, 0, 1).contiguous()

def make_masks(rng, height, width, count, coverage):
    # Rectangles target equal shares of the requested total area; coordinates are persisted.
    area_each = height * width * coverage / count
    masks = []
    for _ in range(count):
        aspect = rng.uniform(0.5, 2.0)
        rw = max(4, min(width, int(math.sqrt(area_each * aspect))))
        rh = max(4, min(height, int(math.sqrt(area_each / aspect))))
        top = rng.randint(0, height - rh)
        left = rng.randint(0, width - rw)
        masks.append({"top": top, "left": left, "height": rh, "width": rw})
    return masks

def sample_train_params(rng):
    kind = rng.choice(CLASSES)
    params = {"corruption_type": kind, "severity": "train", "seed": rng.randrange(2**31)}
    if kind == "salt": params["prob"] = rng.uniform(0.02, 0.15)
    elif kind == "blur": params.update(kernel_size=rng.choice([3, 5, 7]), sigma=rng.uniform(0.5, 2.5))
    elif kind == "occlusion":
        params["num_rects"] = rng.randint(1, 3); params["coverage"] = rng.uniform(0.10, 0.35)
        params["masks"] = make_masks(rng, IMAGE_SIZE, IMAGE_SIZE, params["num_rects"], params["coverage"])
    return params

def apply_corruption(clean, params):
    kind = params["corruption_type"]
    if kind == "clean": return clean.clone()
    if kind == "salt":
        gen = torch.Generator(device="cpu").manual_seed(int(params["seed"]))
        mask = torch.rand((1, *clean.shape[-2:]), generator=gen) < params["prob"]
        salt = torch.rand((1, *clean.shape[-2:]), generator=gen) < 0.5
        out = clean.clone()
        out = torch.where(mask & salt, torch.ones_like(out), out)
        return torch.where(mask & ~salt, torch.zeros_like(out), out)
    if kind == "blur":
        k, sigma = int(params["kernel_size"]), float(params["sigma"])
        axis = torch.arange(k, dtype=clean.dtype) - (k - 1) / 2
        kernel1d = torch.exp(-0.5 * (axis / sigma).square()); kernel1d /= kernel1d.sum()
        kernel2d = torch.outer(kernel1d, kernel1d).view(1, 1, k, k).repeat(3, 1, 1, 1)
        return F.conv2d(clean.unsqueeze(0), kernel2d, padding=k // 2, groups=3).squeeze(0).clamp(0, 1)
    if kind == "occlusion":
        out = clean.clone()
        for m in params["masks"]:
            t, l, h, w = m["top"], m["left"], m["height"], m["width"]
            out[:, t:t+h, l:l+w] = 0.0
        return out
    raise ValueError(f"Unknown corruption: {kind}")

def make_validation_manifest(names):
    manifest = {}
    for i, name in enumerate(names):
        rng = random.Random(SEED + 100_000 + i)
        params = sample_train_params(rng)
        manifest[name] = params
    return manifest

def make_test_manifest(names):
    manifest = {}
    for i, name in enumerate(names):
        rows = [{"corruption_type": "clean", **BENCHMARKS["clean"][0], "seed": SEED + i}]
        for kind in ("salt", "blur", "occlusion"):
            for j, setting in enumerate(BENCHMARKS[kind]):
                params = {"corruption_type": kind, **setting, "seed": SEED + i * 31 + j}
                if kind == "occlusion":
                    params["masks"] = make_masks(random.Random(params["seed"]), IMAGE_SIZE, IMAGE_SIZE,
                                                params["num_rects"], params["coverage"])
                rows.append(params)
        manifest[name] = rows
    return manifest

class TrainPetDataset(Dataset):
    def __init__(self, names): self.names = names
    def __len__(self): return len(self.names)
    def __getitem__(self, idx):
        clean = image_tensor(self.names[idx])
        params = sample_train_params(random)
        return apply_corruption(clean, params), clean

class FixedPetDataset(Dataset):
    def __init__(self, names, manifest):
        self.rows = [(name, params) for name in names for params in (manifest[name] if isinstance(manifest[name], list) else [manifest[name]])]
    def __len__(self): return len(self.rows)
    def __getitem__(self, idx):
        name, params = self.rows[idx]
        clean = image_tensor(name)
        return apply_corruption(clean, params), clean, name, params["corruption_type"], params["severity"]

train_names = SPLITS["train"][:TINY_SIZE] if TINY_RUN else SPLITS["train"]
val_names = SPLITS["val"][:TINY_SIZE] if TINY_RUN else SPLITS["val"]
test_names = SPLITS["test"][:TINY_SIZE] if TINY_RUN else SPLITS["test"]
val_manifest = make_validation_manifest(val_names)
test_manifest = make_test_manifest(test_names)
MANIFEST_DIR = RESEARCH_ROOT / "manifests"
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)
VAL_MANIFEST_PATH = MANIFEST_DIR / "task1_standalone_val_manifest.json"
TEST_MANIFEST_PATH = MANIFEST_DIR / "task1_standalone_test_manifest.json"
VAL_MANIFEST_PATH.write_text(json.dumps(val_manifest, indent=2))
TEST_MANIFEST_PATH.write_text(json.dumps(test_manifest, indent=2))
print("Saved deterministic manifests:", VAL_MANIFEST_PATH.name, TEST_MANIFEST_PATH.name)

train_ds = TrainPetDataset(train_names)
val_ds = FixedPetDataset(val_names, val_manifest)
test_ds = FixedPetDataset(test_names, test_manifest)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
print(f"Train {len(train_ds):,} | validation cases {len(val_ds):,} | test cases {len(test_ds):,}")

## 3. Inspect generated input/target pairs

In [ ]:
batch_noisy, batch_clean = next(iter(train_loader))
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i in range(4):
    axes[0, i].imshow(batch_noisy[i].permute(1, 2, 0)); axes[0, i].axis("off"); axes[0, i].set_title("Input")
    axes[1, i].imshow(batch_clean[i].permute(1, 2, 0)); axes[1, i].axis("off"); axes[1, i].set_title("Clean target")
plt.tight_layout()

## 4. Universal convolutional autoencoder

The encoder progressively reduces spatial size to a genuine fully connected latent vector. It has no skip connections, so reconstruction depends on the compressed representation.

In [ ]:
class ConvDAE(nn.Module):
    def __init__(self, base_channels=32, latent_dim=256, dropout=0.1):
        super().__init__()
        c1, c2, c3, c4, c5 = base_channels, base_channels*2, base_channels*4, base_channels*8, base_channels*8
        def block(cin, cout, drop=True):
            return nn.Sequential(nn.Conv2d(cin, cout, 4, stride=2, padding=1), nn.GroupNorm(8, cout),
                                 nn.LeakyReLU(0.2, inplace=True), nn.Dropout2d(dropout) if drop and dropout else nn.Identity())
        self.enc1, self.enc2, self.enc3 = block(3,c1), block(c1,c2), block(c2,c3)
        self.enc4, self.enc5 = block(c3,c4,False), block(c4,c5,False)
        self.fc_z = nn.Linear(c5*4*4, latent_dim)
        self.fc_dec = nn.Linear(latent_dim, c5*4*4)
        def up(cin, cout, final=False):
            layers = [nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
                      nn.Conv2d(cin, cout, 3, padding=1)]
            if final: layers.append(nn.Sigmoid())
            else: layers.extend([nn.GroupNorm(8, cout), nn.LeakyReLU(0.2, inplace=True)])
            return nn.Sequential(*layers)
        self.dec5, self.dec4, self.dec3 = up(c5,c4), up(c4,c3), up(c3,c2)
        self.dec2, self.dec1 = up(c2,c1), up(c1,3,final=True)
        self.c5 = c5
    def encode(self, x):
        x=self.enc1(x); x=self.enc2(x); x=self.enc3(x); x=self.enc4(x); x=self.enc5(x)
        return self.fc_z(torch.flatten(x, 1))
    def forward(self, x):
        z=self.encode(x); x=self.fc_dec(z).view(-1,self.c5,4,4)
        return self.dec1(self.dec2(self.dec3(self.dec4(self.dec5(x)))))

model = ConvDAE(BASE_CHANNELS, LATENT_DIM, DROPOUT).to(device)
with torch.inference_mode():
    sample = torch.zeros(2, 3, IMAGE_SIZE, IMAGE_SIZE, device=device)
    print("latent:", tuple(model.encode(sample).shape), "output:", tuple(model(sample).shape))
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 5. Loss and training helpers

The objective follows the assignment: `alpha * L1 + (1-alpha) * (1-SSIM)`. Validation averages this objective over a deterministic, balanced set spanning all four conditions.

In [ ]:
def dae_loss(pred, target, alpha):
    l1 = F.l1_loss(pred, target)
    ssim = structural_similarity_index_measure(pred, target, data_range=1.0)
    return alpha * l1 + (1.0-alpha) * (1.0-ssim)

def run_epoch(net, loader, optimizer=None, alpha=ALPHA):
    training = optimizer is not None
    net.train(training); total = 0.0; count = 0
    with torch.set_grad_enabled(training):
        for noisy, clean, *_ in loader:
            noisy, clean = noisy.to(device, non_blocking=True), clean.to(device, non_blocking=True)
            pred = net(noisy); loss = dae_loss(pred, clean, alpha)
            if training:
                optimizer.zero_grad(set_to_none=True); loss.backward(); optimizer.step()
            total += loss.item() * clean.size(0); count += clean.size(0)
    return total / max(1, count)

def fit(net, train_dl, valid_dl, epochs, lr, alpha, save_path=None):
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    best, best_state, history = float("inf"), None, {"train": [], "val": []}
    for epoch in range(1, epochs+1):
        tr = run_epoch(net, train_dl, opt, alpha)
        va = run_epoch(net, valid_dl, None, alpha)
        history["train"].append(tr); history["val"].append(va)
        if va < best:
            best, best_state = va, copy.deepcopy(net.state_dict())
            if save_path: torch.save(best_state, save_path)
        print(f"Epoch {epoch:02d}/{epochs}: train={tr:.4f} val={va:.4f}")
    if best_state is not None: net.load_state_dict(best_state)
    return history, best

## 6. Optuna hyperparameter search

The search covers learning rate, batch size, latent dimension, encoder width, dropout, and loss weight `alpha`. Trial validation uses the fixed validation manifest. Set the trial count and epoch counts above to match available compute; report the completed trial count and selected parameters.

In [ ]:
def objective(trial):
    params = {
        "lr": trial.suggest_float("lr", 1e-4, 1e-3, log=True),
        "batch_size": trial.suggest_categorical("batch_size", [16, 32, 64]),
        "latent_dim": trial.suggest_categorical("latent_dim", [64, 128, 256, 512]),
        "base_channels": trial.suggest_categorical("base_channels", [16, 32, 64]),
        "dropout": trial.suggest_float("dropout", 0.0, 0.3),
        "alpha": trial.suggest_float("alpha", 0.5, 0.95),
    }
    trial_train_loader = DataLoader(train_ds, batch_size=params["batch_size"], shuffle=True,
                                    num_workers=NUM_WORKERS, pin_memory=True)
    net = ConvDAE(params["base_channels"], params["latent_dim"], params["dropout"]).to(device)
    _, best_val = fit(net, trial_train_loader, val_loader, SEARCH_EPOCHS, params["lr"], params["alpha"])
    del net
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return best_val

study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED),
                            pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=2))
study.optimize(objective, n_trials=N_TRIALS)
print("Completed:", len(study.get_trials(states=(optuna.trial.TrialState.COMPLETE,))))
print("Best trial:", study.best_trial.number, "| validation objective:", study.best_value)
print("Selected configuration:", study.best_params)

## 7. Retrain the selected configuration

In [ ]:
best_params = study.best_params
BATCH_SIZE = best_params["batch_size"]
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
model = ConvDAE(best_params["base_channels"], best_params["latent_dim"], best_params["dropout"]).to(device)
CHECKPOINT_DIR = RESEARCH_ROOT / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = CHECKPOINT_DIR / "task1_standalone_universal_dae_best.pt"
history, best_val = fit(model, train_loader, val_loader, FINAL_EPOCHS, best_params["lr"], best_params["alpha"], CHECKPOINT_PATH)
print("Best validation objective:", best_val, "| checkpoint:", CHECKPOINT_PATH)

fig, ax = plt.subplots(figsize=(7,4))
ax.plot(history["train"], label="train"); ax.plot(history["val"], label="validation")
ax.set(xlabel="Epoch", ylabel="Composite loss", title="Task 1 training curves"); ax.legend(); ax.grid(True, alpha=.3)
plt.show()

## 8. Held-out test evaluation by corruption and severity

PSNR and SSIM are calculated separately for clean inputs and each corruption severity. The test images are not used for optimization or checkpoint selection.

In [ ]:
@torch.inference_mode()
def evaluate_by_condition(net, loader):
    net.eval(); sums = {}
    for noisy, clean, names, kinds, severities in loader:
        noisy, clean = noisy.to(device), clean.to(device)
        pred = net(noisy).clamp(0,1)
        mse = (pred-clean).square().flatten(1).mean(1).clamp_min(1e-10)
        psnr = 10*torch.log10(1.0/mse)
        # Compute per-image SSIM so results can be grouped by each corruption case.
        ssim = torch.stack([structural_similarity_index_measure(pred[i:i+1], clean[i:i+1], data_range=1.0)
                            for i in range(len(pred))])
        for i, key in enumerate(zip(kinds, severities)):
            if key not in sums: sums[key] = [0.0, 0.0, 0]
            sums[key][0] += float(psnr[i]); sums[key][1] += float(ssim[i]); sums[key][2] += 1
    return [{"corruption": k[0], "severity": k[1], "PSNR": v[0]/v[2], "SSIM": v[1]/v[2], "n": v[2]}
            for k,v in sorted(sums.items())]

results = evaluate_by_condition(model, test_loader)
for row in results: print(f"{row['corruption']:10s} {row['severity']:7s} | PSNR {row['PSNR']:.2f} dB | SSIM {row['SSIM']:.4f} | n={row['n']}")
RESULTS_PATH = RESEARCH_ROOT / "reports" / "task1_standalone_test_metrics.json"
RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
RESULTS_PATH.write_text(json.dumps({"best_params": best_params, "best_validation_objective": best_val, "metrics": results}, indent=2))
print("Saved:", RESULTS_PATH)

## 9. Twelve representative reconstructions with absolute-error maps

In [ ]:
@torch.inference_mode()
def collect_examples(net, dataset, count=12):
    net.eval(); indices = np.linspace(0, len(dataset)-1, min(count, len(dataset)), dtype=int)
    rows=[]
    for idx in indices:
        noisy, clean, name, kind, severity = dataset[int(idx)]
        pred = net(noisy.unsqueeze(0).to(device)).squeeze(0).cpu().clamp(0,1)
        rows.append((name, kind, severity, clean, noisy, pred, (clean-pred).abs().mean(0)))
    return rows

examples = collect_examples(model, test_ds, 12)
fig, axes = plt.subplots(len(examples), 4, figsize=(12, 3*len(examples)), squeeze=False)
for i, (name, kind, severity, clean, noisy, pred, err) in enumerate(examples):
    for j, (im, title) in enumerate([(clean,"Clean target"),(noisy,f"{kind} / {severity}"),(pred,"Reconstruction"),(err,"Mean absolute error")]):
        axes[i,j].imshow(im.permute(1,2,0) if im.ndim==3 else im, cmap="magma" if j==3 else None, vmin=0 if j==3 else None, vmax=1 if j==3 else None)
        axes[i,j].axis("off")
        if i==0: axes[i,j].set_title(title)
    axes[i,0].set_ylabel(name, rotation=0, labelpad=38, fontsize=8)
plt.tight_layout(); plt.show()

## 10. Inspect four worst test cases

These examples provide concrete failure cases for the report. Interpret the visible failure (for example, lost fur detail, residual impulse artifacts, over-smoothed edges, or content hidden by an occlusion) alongside the printed corruption and severity.

In [ ]:
@torch.inference_mode()
def worst_cases(net, dataset, n=4):
    net.eval(); scored=[]
    # Deterministic subset across the complete test benchmark for quick failure analysis.
    indices = np.linspace(0, len(dataset)-1, min(len(dataset), max(200, n)), dtype=int)
    for idx in indices:
        noisy, clean, name, kind, severity = dataset[int(idx)]
        pred=net(noisy.unsqueeze(0).to(device)).squeeze(0).cpu().clamp(0,1)
        mse=float((pred-clean).square().mean())
        scored.append((mse,name,kind,severity,clean,noisy,pred,(clean-pred).abs().mean(0)))
    return sorted(scored, reverse=True, key=lambda x:x[0])[:n]

failures=worst_cases(model,test_ds,4)
fig,axes=plt.subplots(len(failures),4,figsize=(12,3*len(failures)),squeeze=False)
for i,(mse,name,kind,severity,clean,noisy,pred,err) in enumerate(failures):
    print(f"Failure {i+1}: {name} | {kind}/{severity} | MSE={mse:.5f}")
    for j,im in enumerate((clean,noisy,pred,err)):
        axes[i,j].imshow(im.permute(1,2,0) if im.ndim==3 else im,cmap="magma" if j==3 else None,vmin=0 if j==3 else None,vmax=1 if j==3 else None)
        axes[i,j].axis("off")
        if i==0: axes[i,j].set_title(("Clean","Corrupted","Restored","Absolute error")[j])
plt.tight_layout(); plt.show()

## 11. Reproducibility and run summary

The notebook writes deterministic validation/test manifests, a selected-model checkpoint, and a JSON metric report. Preserve the manifests and the completed Optuna trial count with the report so the evaluation can be reproduced.